# Transformer Encoder — IRB2400 Kinematics (Baseline, no Positional Encoding)

## 이 노트북이 하는 일
IRB2400 6자유도 산업용 로봇팔에서
- **입력**: 과거 10 스텝의 관절각 명령 `q1~q6`
- **출력**: 다음 시점의 엔드이펙터 포즈 `(x, y, z, yaw, pitch, roll)`
를 예측하는 **시퀀스-투-벡터 회귀** 모델이야.

## 이 버전의 특징 (baseline)
- **Positional Encoding 이 없다** — 순수 Transformer Encoder 로 얼마나 가는지 확인하는 대조군
- Encoder 2블록 (self-attention + FFN) → GlobalAveragePooling → Dense(6)

PE 를 넣은 버전은 `tr_encoder_kinematics_pe.ipynb` 에 있고, 이 노트북과 직접 비교하기 위한 베이스라인이다.


---
## [1] 라이브러리 임포트

- `StandardScaler` : 입·출력 각 차원의 스케일을 맞춰서 MSE 학습 안정화
- `MultiHeadAttention` / `LayerNormalization` / `Add` : Transformer 블록 빌딩 블록
- `GlobalAveragePooling1D` : 시간축 평균으로 (B, T, d) → (B, d)
- `mean_absolute_error` : 회귀 손실을 실제 단위로 해석하기 위한 평가지표


In [ ]:
# [1] 라이브러리 임포트
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler           # z-score 정규화 (MSE 기반 회귀에서 필수급)
from sklearn.metrics import mean_absolute_error            # 평균 절대 오차 (해석 쉬움)

import tensorflow as tf
from tensorflow.keras.models import Model                  # Functional API — 커스텀 블록에 유연
from tensorflow.keras.layers import (
    Input, Dense, LayerNormalization, Dropout,
    GlobalAveragePooling1D,                                # (B,T,d)->(B,d) 시간축 평균
    MultiHeadAttention, Add,                               # Transformer 의 핵심 + 잔차 연결
)


---
## [2] Google Drive 마운트

Colab 에서 학습 CSV 를 읽기 위해 Drive 마운트. 로컬에서 돌릴 땐 이 셀 건너뛰고 `csv_path` 만 바꾸면 돼.


In [ ]:
# [2] Google Drive 마운트 (Colab 전용)
from google.colab import drive
drive.mount('/content/drive')


---
## [3] 데이터 로딩

`datasetIRB2400.csv` 는 시간 순서대로 쌓인 (관절각, 포즈) 샘플들이 있어야 함.
`df.head()` 로 스케일과 dtype 을 눈으로 확인 — 정규화 필요성을 체감하는 용도.


In [ ]:
# [3] 데이터 로딩
csv_path = '/content/drive/MyDrive/Colab Notebooks/data/ngv/datasetIRB2400.csv'
df = pd.read_csv(csv_path)
print("데이터셋 로딩 완료")
print(df.head())   # 컬럼 이름과 값 범위 확인용


---
## [4]~[5] 입/출력 + 정규화

### 왜 StandardScaler?
- 위치(mm) vs 각도(rad) 처럼 **단위/스케일 차이가 큼** → MSE 가 특정 차원에만 끌려감
- z-score 정규화로 평균 0·분산 1 로 맞추면 attention dot-product 분포도 안정

### 왜 X, Y 각각 따로?
- 입력/타깃은 의미가 달라서 같은 scaler 로 합쳐서 fit 하면 안 됨
- 예측값 복원도 `scaler_y.inverse_transform` 하나로 깔끔하게 처리 가능


In [ ]:
# [4] 입력 및 출력 설정
X = df[['q1_in', 'q2_in', 'q3_in', 'q4_in', 'q5_in', 'q6_in']].values   # 관절각 (N, 6)
y = df[['x', 'y', 'z', 'yaw', 'pitch', 'roll']].values                   # 포즈   (N, 6)

# [5] 정규화 (입·출력 각각 따로 fit)
scaler_X = StandardScaler()
scaler_y = StandardScaler()
X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y)


---
## [6] 시퀀스 생성 — sliding window

시계열을 지도학습 문제로 바꾸려면 (과거 T스텝, 다음 1스텝) 쌍을 만들어야 함.

```
시간축 →
[q_0, q_1, ..., q_{T-1}]  →  y_T     (sample 0)
[q_1, q_2, ..., q_T    ]  →  y_{T+1} (sample 1)
...
```

### 주의: 원본 코드의 '숨겨진 버그'
원본은 `def create_sequences(X, y, time_steps=40)` 으로 **함수 기본값이 40** 인데,
실제 호출은 `time_steps = 10` 으로 바깥 변수를 넘겨서 **10** 으로 돈다.
여기서는 기본값도 10 으로 맞춰서 헷갈림을 없앰.


In [ ]:
# [6] 시퀀스 생성 (sliding window)
def create_sequences(X, y, time_steps=10):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:i + time_steps])      # 과거 T 스텝 관절각 → (T, 6)
        ys.append(y[i + time_steps])        # 바로 다음 시점 포즈 → (6,)
    return np.array(Xs), np.array(ys)

time_steps = 10                              # 짧은 윈도우 — baseline 용 간단한 설정
X_seq, y_seq = create_sequences(X_scaled, y_scaled, time_steps)


---
## [7] 학습/검증 분리

- `test_size=0.2` : 80/20
- `random_state=42` : 재현성
- 시계열이지만 기존 노트북과 동일한 random split 을 유지 (비교를 위해)


In [ ]:
# [7] 학습/테스트 분리
X_train, X_test, y_train, y_test = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)
print(f"Train shape: {X_train.shape}, Test shape: {X_test.shape}")


---
## [8] Transformer Encoder Block

### 구조 (Pre-Norm)
```
x ─┬─ LN ─ MHA(self) ─┬── +x ─┬─ LN ─ Dense(ff,relu) ─ Dense(d) ─┬── +x ── out
   └── residual ──────┘      └───────── residual ───────────────┘
```

### 왜 Pre-Norm 인가
- 원 논문은 Post-Norm 이지만 깊이가 깊어질수록 gradient 가 불안정
- Pre-Norm 은 residual path 가 raw 입력을 유지 → 학습 안정

### 왜 residual connection 인가
- 깊은 네트워크 gradient 보존
- Attention/FFN 은 "입력에 대한 변화량" 만 학습하면 됨

### 주의
이 baseline 은 **PE 가 없기 때문에** self-attention 이 시퀀스 순서를 직접 구분하지 못함.
그럼에도 유의미하게 학습되는 이유는 (a) 각 타임스텝의 값 자체가 서로 구분되고
(b) Average Pooling 이 시간 정보를 요약해 주기 때문. 그래도 일반적으로 PE 버전이 더 잘 된다.


In [ ]:
# [8] Transformer Encoder 정의
def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # ---- Self-Attention sub-layer (Pre-Norm) ----
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(                                  # Q=K=V=x → self-attention
        key_dim=head_size,                                   # head 당 key/query 차원
        num_heads=num_heads,                                 # 여러 '관점'으로 attend
        dropout=dropout,
    )(x, x)                                                  # mask 없음 = bidirectional
    x = Add()([x, inputs])                                   # residual

    # ---- Feed-Forward sub-layer ----
    x = LayerNormalization(epsilon=1e-6)(x)
    x_ff = Dense(ff_dim, activation='relu')(x)               # 확장 (표현력↑)
    x_ff = Dense(inputs.shape[-1])(x_ff)                     # 다시 d_model 로 투영
    return Add()([x, x_ff])                                  # residual


---
## [9] 모델 구성 (Encoder-only)

### 하이퍼파라미터 의미
- `head_size=64, num_heads=4` → attention 총 차원 256 (입력 6차원에 비해 충분히 큼)
- `ff_dim=128` → FFN hidden. 표현력 보강
- `dropout=0.1` → 과적합 완화

### 회귀 head
- `GlobalAveragePooling1D` : 시간축 평균 → (B, d_model). 마지막 토큰만 쓰는 것보다 안정
- `Dense(64, relu) → Dense(6)` : 단순 MLP head. 최종 출력 activation 없음(선형) — 회귀라서


In [ ]:
# [9] 모델 구성 (Encoder-only)
input_shape = (time_steps, X_train.shape[2])                  # (10, 6)
inputs = Input(shape=input_shape)

# Encoder 2 블록
x = transformer_encoder(inputs, head_size=64, num_heads=4, ff_dim=128, dropout=0.1)
x = transformer_encoder(x,      head_size=64, num_heads=4, ff_dim=128, dropout=0.1)

# 회귀 head
x = GlobalAveragePooling1D()(x)                               # (B, T, d) → (B, d)
x = Dense(64, activation='relu')(x)
outputs = Dense(6)(x)                                         # 선형 출력 — 포즈 회귀

model = Model(inputs, outputs)
model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


---
## [10] 학습

- `epochs=10` : baseline 이라 짧게
- `batch_size=32` : 데이터가 많지 않아 작은 배치로 gradient noise 확보
- `validation_data=(X_test, y_test)` : 간단히 test 를 val 로 겸용 (baseline 비교 용)
- EarlyStopping 없이 고정 epoch — PE 버전과 "동일 조건" 을 맞추기 위해


In [ ]:
# [10] 모델 학습
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32,
)


---
## [11] 학습 곡선 시각화

train loss 와 val loss 를 같이 보면서
- 둘 다 감소 → 정상
- train 만 감소, val 증가 → 과적합 조짐
을 판단.


In [ ]:
# [11] 학습 시각화
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.grid(True)
plt.title("Training/Validation Loss")
plt.show()


---
## [12]~[14] 예측 · 역정규화 · MAE

모델은 정규화된 공간에서 학습했으므로 예측도 정규화 공간의 값이 나옴.
- `scaler_y.inverse_transform` 으로 실제 단위 (mm, rad) 로 복원
- 정답도 동일하게 복원해야 **같은 공간에서 오차 비교**


In [ ]:
# [12] 예측
y_pred_scaled = model.predict(X_test)

# [13] 역정규화
y_pred = scaler_y.inverse_transform(y_pred_scaled)
y_true = scaler_y.inverse_transform(y_test)

# [14] MAE (전체)
mae_real = mean_absolute_error(y_true, y_pred)
print(f"실제 단위 기준 MAE: {mae_real:.4f}")


---
## [15] 차원별 MAE

전체 MAE 는 6차원 평균이라 어떤 차원이 특히 취약한지 숨겨져 있음.
`x, y, z` (mm) 과 `yaw, pitch, roll` (rad) 는 **단위 자체가 달라서** 각각 따로 보는 게 의미 있음.


In [ ]:
# [15] 각 항목별 MAE
columns = ['x', 'y', 'z', 'yaw', 'pitch', 'roll']
mae_each = np.mean(np.abs(y_true - y_pred), axis=0)
for name, value in zip(columns, mae_each):
    print(f"{name} MAE: {value:.4f}")
